In [25]:
# ============================================================
# YT8M — Проверка содержимого TFRecord файлов
# ============================================================
# pip install tensorflow-cpu  (если не установлен)
import tensorflow as tf
import numpy as np
from pathlib import Path

DATA_DIR = Path(Path('data2/video/validate'))
tfrecord_files = sorted(DATA_DIR.glob('*.tfrecord'))

print(f"📂 Найдено файлов: {len(tfrecord_files)}")
for f in tfrecord_files:
    size_mb = f.stat().st_size / 1024**2
    print(f"   {f.name}  {size_mb:.1f} MB")

📂 Найдено файлов: 362
   validate0012.tfrecord  1.4 MB
   validate0015.tfrecord  1.3 MB
   validate0018.tfrecord  1.4 MB
   validate0020.tfrecord  1.5 MB
   validate0024.tfrecord  1.4 MB
   validate0026.tfrecord  1.3 MB
   validate0032.tfrecord  1.3 MB
   validate0040.tfrecord  1.1 MB
   validate0052.tfrecord  1.4 MB
   validate0065.tfrecord  1.1 MB
   validate0067.tfrecord  1.1 MB
   validate0094.tfrecord  1.3 MB
   validate0107.tfrecord  1.3 MB
   validate0111.tfrecord  1.3 MB
   validate0119.tfrecord  1.4 MB
   validate0125.tfrecord  1.4 MB
   validate0128.tfrecord  1.4 MB
   validate0130.tfrecord  1.3 MB
   validate0135.tfrecord  1.3 MB
   validate0156.tfrecord  1.3 MB
   validate0160.tfrecord  1.3 MB
   validate0166.tfrecord  1.3 MB
   validate0170.tfrecord  1.4 MB
   validate0188.tfrecord  1.3 MB
   validate0210.tfrecord  1.3 MB
   validate0211.tfrecord  1.3 MB
   validate0212.tfrecord  1.4 MB
   validate0215.tfrecord  1.3 MB
   validate0225.tfrecord  1.3 MB
   validate0233.tfrec

In [26]:
# ── Читаем один файл, смотрим структуру ──────────────────────
print("🔍 Структура одной записи:")
dataset = tf.data.TFRecordDataset(str(tfrecord_files[0]), compression_type='')
for raw in dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw.numpy())
    for key, value in sorted(example.features.feature.items()):
        kind = value.WhichOneof('kind')
        if kind == 'int64_list':
            vals = value.int64_list.value
            print(f"   {key:<20} int64   len={len(vals)}  sample={list(vals[:5])}")
        elif kind == 'float_list':
            vals = value.float_list.value
            print(f"   {key:<20} float   len={len(vals)}  sample={[round(v,3) for v in vals[:5]]}")
        elif kind == 'bytes_list':
            vals = value.bytes_list.value
            print(f"   {key:<20} bytes   len={len(vals)}")

🔍 Структура одной записи:
   id                   bytes   len=1
   labels               int64   len=4  sample=[0, 1, 36, 185]
   mean_audio           float   len=128  sample=[-0.842, -0.22, -0.89, 0.687, -1.305]
   mean_rgb             float   len=1024  sample=[0.888, -1.794, -0.556, 0.721, 0.103]


In [27]:
# ── Считаем сколько всего видео ───────────────────────────────
print("📊 Подсчёт видео по файлам:")
total = 0
for tf_file in tfrecord_files:
    ds  = tf.data.TFRecordDataset(str(tf_file))
    cnt = sum(1 for _ in ds)
    total += cnt
    print(f"   {tf_file.name}  →  {cnt} видео")
print(f"\n   ИТОГО: {total} видео")

📊 Подсчёт видео по файлам:
   validate0012.tfrecord  →  322 видео
   validate0015.tfrecord  →  293 видео
   validate0018.tfrecord  →  309 видео
   validate0020.tfrecord  →  342 видео
   validate0024.tfrecord  →  322 видео
   validate0026.tfrecord  →  292 видео
   validate0032.tfrecord  →  282 видео
   validate0040.tfrecord  →  254 видео
   validate0052.tfrecord  →  312 видео
   validate0065.tfrecord  →  252 видео
   validate0067.tfrecord  →  243 видео
   validate0094.tfrecord  →  293 видео
   validate0107.tfrecord  →  285 видео
   validate0111.tfrecord  →  280 видео
   validate0119.tfrecord  →  320 видео
   validate0125.tfrecord  →  304 видео
   validate0128.tfrecord  →  308 видео
   validate0130.tfrecord  →  284 видео
   validate0135.tfrecord  →  287 видео
   validate0156.tfrecord  →  299 видео
   validate0160.tfrecord  →  300 видео
   validate0166.tfrecord  →  289 видео
   validate0170.tfrecord  →  303 видео
   validate0188.tfrecord  →  296 видео
   validate0210.tfrecord  →  289 виде

In [28]:
# ============================================================
# YT8M — Правильный словарь меток
# ============================================================
import urllib.request, csv
from pathlib import Path
from collections import Counter

# Правильный URL из официальной документации YT8M
vocab_url  = 'http://research.google.com/youtube8m/csv/segments/vocabulary.csv'
vocab_path = Path(Path('data2')) / 'vocabulary.csv'

print("📥 Скачиваем vocabulary.csv...", end=' ')
urllib.request.urlretrieve(vocab_url, vocab_path)
print("✅")

# Смотрим структуру первых строк
print("\n🔍 Первые 5 строк:")
with open(vocab_path, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"   {repr(line.strip())}")
        if i >= 4:
            break

# Читаем словарь
label_vocab = {}   # {int_index → name}
with open(vocab_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    print(f"\n   Колонки: {reader.fieldnames}")
    for row in reader:
        # Первая колонка — числовой индекс (Index)
        idx  = int(row.get('Index', row.get('index', -1)))
        name = row.get('Name', row.get('name', row.get('KnowledgeGraphId', '?')))
        if idx >= 0:
            label_vocab[idx] = name

print(f"\n📖 Всего меток: {len(label_vocab)}")
print(f"   Примеры: { {k: label_vocab[k] for k in list(label_vocab)[:8]} }")

📥 Скачиваем vocabulary.csv... ✅

🔍 Первые 5 строк:
   'Index,KnowledgeGraphId,Name,PositiveSegmentCount,TotalSegmentCount'
   '3,/m/01jddz,Concert,181,240'
   '7,/m/0k4j,Car,159,246'
   '8,/m/026bk,Dance,200,245'
   '11,/m/02wbm,Food,121,247'

   Колонки: ['Index', 'KnowledgeGraphId', 'Name', 'PositiveSegmentCount', 'TotalSegmentCount']

📖 Всего меток: 142
   Примеры: {3: 'Concert', 7: 'Car', 8: 'Dance', 11: 'Food', 12: 'Association football', 17: 'Motorsport', 18: 'Pet', 19: 'Racing'}


In [29]:
# ============================================================
# YT8M — Правильный словарь меток (3862 класса, video-level)
# ============================================================
import urllib.request, csv
from pathlib import Path
from collections import Counter

# Словарь из репозитория yash1994 — проверенный источник для 2018 версии
vocab_url  = 'https://raw.githubusercontent.com/yash1994/youtube-8m-videos-downloader/master/vocabulary.csv'
vocab_path = Path(Path('data2')) / 'vocabulary_full.csv'

print("📥 Скачиваем полный словарь...", end=' ')
urllib.request.urlretrieve(vocab_url, vocab_path)
print("✅")

# Смотрим структуру
print("\n🔍 Первые 5 строк:")
with open(vocab_path, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"   {repr(line.strip())}")
        if i >= 4:
            break

# Читаем словарь
label_vocab = {}
with open(vocab_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    print(f"\n   Колонки: {reader.fieldnames}")
    for row in reader:
        # Берём первую числовую колонку как index
        for key in row:
            try:
                idx = int(row[key])
                # Берём следующую строковую колонку как имя
                vals = list(row.values())
                name_candidates = [v for v in vals
                                   if v and not v.lstrip('-').isdigit()
                                   and not v.startswith('/')]
                if name_candidates:
                    label_vocab[idx] = name_candidates[0]
                break
            except (ValueError, TypeError):
                continue

print(f"\n📖 Всего меток: {len(label_vocab)}")
print(f"   Примеры: { {k: label_vocab[k] for k in list(label_vocab)[:8]} }")

📥 Скачиваем полный словарь... ✅

🔍 Первые 5 строк:
   'Index,TrainVideoCount,KnowledgeGraphId,Name'
   '0,788288,/m/03bt1gh,Game'
   '1,539945,/m/01mw1,Video game'
   '2,415890,/m/07yv9,Vehicle'
   '3,378135,/m/01jddz,Concert'

   Колонки: ['Index', 'TrainVideoCount', 'KnowledgeGraphId', 'Name']

📖 Всего меток: 3806
   Примеры: {0: 'Game', 1: 'Video game', 2: 'Vehicle', 3: 'Concert', 4: 'Musician', 5: 'Cartoon', 6: 'Performance art', 7: 'Car'}


In [30]:
# ============================================================
# YT8M — Подсчёт частоты меток и выбор жанров
# ============================================================
from collections import Counter

print("🔄 Сканируем все TFRecord файлы...")

label_counter    = Counter()
multilabel_count = 0
total_videos     = 0

for tf_file in tfrecord_files:
    ds = tf.data.TFRecordDataset(str(tf_file))
    for raw in ds:
        example = tf.train.Example()
        example.ParseFromString(raw.numpy())
        lbls = list(example.features.feature['labels'].int64_list.value)
        label_counter.update(lbls)
        if len(lbls) > 1:
            multilabel_count += 1
        total_videos += 1

print(f"📊 Итого видео      : {total_videos}")
print(f"   Мультилейбл     : {multilabel_count} ({multilabel_count/total_videos*100:.1f}%)")
print(f"   Уникальных меток: {len(label_counter)}")

# ── Топ-40 самых частых классов ───────────────────────────────
print(f"\n🏆 Топ-40 самых частых классов:")
print(f"   {'ID':>5}  {'Название':<35}  {'Кол-во':>7}  {'%':>5}")
print("   " + "─" * 58)
for label_id, count in label_counter.most_common(40):
    name = label_vocab.get(label_id, f'unknown_{label_id}')
    pct  = count / total_videos * 100
    print(f"   {label_id:>5}  {name:<35}  {count:>7}  {pct:>4.1f}%")

🔄 Сканируем все TFRecord файлы...
📊 Итого видео      : 105147
   Мультилейбл     : 79207 (75.3%)
   Уникальных меток: 3853

🏆 Топ-40 самых частых классов:
      ID  Название                              Кол-во      %
   ──────────────────────────────────────────────────────────
       0  Game                                   21282  20.2%
       1  Video game                             14444  13.7%
       2  Vehicle                                11343  10.8%
       3  Concert                                10302   9.8%
       4  Musician                                7690   7.3%
       5  Cartoon                                 6517   6.2%
       7  Car                                     5601   5.3%
       6  Performance art                         5567   5.3%
       8  Dance                                   5012   4.8%
       9  Guitar                                  4208   4.0%
      10  String instrument                       3931   3.7%
      11  Food                         

In [31]:
# ── Найдём классы с хорошим балансом (200–800 видео) ─────────
print("\n🎯 Классы с 200–800 видео (кандидаты для датасета):")
print(f"   {'ID':>5}  {'Название':<35}  {'Кол-во':>7}")
print("   " + "─" * 52)

candidates = []
for label_id, count in label_counter.most_common():
    if 200 <= count <= 800:
        name = label_vocab.get(label_id, f'unknown_{label_id}')
        candidates.append((label_id, name, count))
        print(f"   {label_id:>5}  {name:<35}  {count:>7}")

print(f"\n   Итого кандидатов: {len(candidates)}")


🎯 Классы с 200–800 видео (кандидаты для датасета):
      ID  Название                              Кол-во
   ────────────────────────────────────────────────────
      57  Musical keyboard                         797
      62  Machine                                  794
      63  Sports game                              789
      60  Outdoor recreation                       781
      58  Bicycle                                  777
      61  Disc jockey                              764
      67  Dog                                      722
      65  Hairstyle                                717
      69  Fighting game                            708
      66  Fashion                                  702
      64  Radio-controlled model                   676
      68  Skateboarding                            653
      75  Truck                                    645
      70  Basketball moves                         633
      74  Personal computer                        629
      76  Bo

In [32]:
# ============================================================
# Пересканируем все файлы после расширенной загрузки
# ============================================================
from pathlib import Path
from collections import Counter
import tensorflow as tf

DATA_DIR       = Path(Path('data2/video/validate'))
tfrecord_files = sorted(DATA_DIR.glob('*.tfrecord'))

print(f"📂 Файлов сейчас: {len(tfrecord_files)}")

label_counter    = Counter()
multilabel_count = 0
total_videos     = 0

for i, tf_file in enumerate(tfrecord_files):
    ds = tf.data.TFRecordDataset(str(tf_file))
    for raw in ds:
        example = tf.train.Example()
        example.ParseFromString(raw.numpy())
        lbls = list(example.features.feature['labels'].int64_list.value)
        label_counter.update(lbls)
        if len(lbls) > 1:
            multilabel_count += 1
        total_videos += 1
    if (i + 1) % 50 == 0:
        print(f"   обработано {i+1}/{len(tfrecord_files)} файлов, видео: {total_videos}")

print(f"\n📊 Итого видео      : {total_videos}")
print(f"   Мультилейбл     : {multilabel_count} ({multilabel_count/total_videos*100:.1f}%)")
print(f"   Уникальных меток: {len(label_counter)}")

📂 Файлов сейчас: 362
   обработано 50/362 файлов, видео: 14683
   обработано 100/362 файлов, видео: 29356
   обработано 150/362 файлов, видео: 43843
   обработано 200/362 файлов, видео: 58500
   обработано 250/362 файлов, видео: 72823
   обработано 300/362 файлов, видео: 87272
   обработано 350/362 файлов, видео: 101669

📊 Итого видео      : 105147
   Мультилейбл     : 79207 (75.3%)
   Уникальных меток: 3853


In [33]:
# ── Топ-50 классов с названиями ───────────────────────────────
print(f"\n🏆 Топ-50 самых частых классов:")
print(f"   {'ID':>5}  {'Название':<35}  {'Кол-во':>8}  {'%':>5}")
print("   " + "─" * 60)
for label_id, count in label_counter.most_common(50):
    name = label_vocab.get(label_id, f'unknown_{label_id}')
    pct  = count / total_videos * 100
    bar  = '█' * int(pct * 2)
    print(f"   {label_id:>5}  {name:<35}  {count:>8}  {pct:>4.1f}%  {bar}")


🏆 Топ-50 самых частых классов:
      ID  Название                               Кол-во      %
   ────────────────────────────────────────────────────────────
       0  Game                                    21282  20.2%  ████████████████████████████████████████
       1  Video game                              14444  13.7%  ███████████████████████████
       2  Vehicle                                 11343  10.8%  █████████████████████
       3  Concert                                 10302   9.8%  ███████████████████
       4  Musician                                 7690   7.3%  ██████████████
       5  Cartoon                                  6517   6.2%  ████████████
       7  Car                                      5601   5.3%  ██████████
       6  Performance art                          5567   5.3%  ██████████
       8  Dance                                    5012   4.8%  █████████
       9  Guitar                                   4208   4.0%  ████████
      10  String inst

In [34]:
# ── Наши 9 планируемых жанров — сколько теперь видео ─────────
GENRE_MAP = {
    'Gaming'    : [0, 1, 35],
    'Music'     : [3, 4, 14, 13, 9, 10, 38],
    'Animation' : [5, 16],
    'Vehicles'  : [2, 7, 17, 19],
    'Sports'    : [12],
    'Animals'   : [15, 18],
    'Food'      : [11, 20, 22],
    'Dance'     : [8],
    'Performance': [6],
}

print(f"\n🎯 Покрытие наших 9 жанров (видео с хотя бы одной меткой жанра):")
print(f"   {'Жанр':<14}  {'YT8M IDs':<25}  {'Видео':>8}")
print("   " + "─" * 55)
total_covered = 0
for genre, ids in GENRE_MAP.items():
    count = sum(label_counter.get(i, 0) for i in ids)
    ids_str = str(ids)
    print(f"   {genre:<14}  {ids_str:<25}  {count:>8}")
    total_covered += count

print(f"\n   Суммарно меток : {total_covered}")
print(f"   Всего видео    : {total_videos}")
print(f"\n   ⚠️  Примечание: одно видео может входить в несколько жанров (мультилейбл)")
print(f"      Реальное число уникальных видео будет меньше суммы")


🎯 Покрытие наших 9 жанров (видео с хотя бы одной меткой жанра):
   Жанр            YT8M IDs                      Видео
   ───────────────────────────────────────────────────────
   Gaming          [0, 1, 35]                    37036
   Music           [3, 4, 14, 13, 9, 10, 38]     33907
   Animation       [5, 16]                        9202
   Vehicles        [2, 7, 17, 19]                21754
   Sports          [12]                           3549
   Animals         [15, 18]                       5507
   Food            [11, 20, 22]                   7691
   Dance           [8]                            5012
   Performance     [6]                            5567

   Суммарно меток : 129225
   Всего видео    : 105147

   ⚠️  Примечание: одно видео может входить в несколько жанров (мультилейбл)
      Реальное число уникальных видео будет меньше суммы


In [35]:
# ── Смотрим классы 51-150 — ищем новые жанры ─────────────────
print(f"📋 Классы с 500+ видео (все, не только топ-50):")
print(f"   {'ID':>5}  {'Название':<35}  {'Кол-во':>8}  {'%':>5}")
print("   " + "─" * 60)

already_mapped = {0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,22,35,38}
candidates_new = []

for label_id, count in label_counter.most_common(200):
    name = label_vocab.get(label_id, f'unknown_{label_id}')
    pct  = count / total_videos * 100
    if count >= 500:
        marker = '  ✅ mapped' if label_id in already_mapped else '  ← NEW?'
        print(f"   {label_id:>5}  {name:<35}  {count:>8}  {pct:>4.1f}%{marker}")
        if label_id not in already_mapped:
            candidates_new.append((label_id, name, count))

print(f"\n🆕 Незамапленных классов с 500+ видео: {len(candidates_new)}")

📋 Классы с 500+ видео (все, не только топ-50):
      ID  Название                               Кол-во      %
   ────────────────────────────────────────────────────────────
       0  Game                                    21282  20.2%  ✅ mapped
       1  Video game                              14444  13.7%  ✅ mapped
       2  Vehicle                                 11343  10.8%  ✅ mapped
       3  Concert                                 10302   9.8%  ✅ mapped
       4  Musician                                 7690   7.3%  ✅ mapped
       5  Cartoon                                  6517   6.2%  ✅ mapped
       7  Car                                      5601   5.3%  ✅ mapped
       6  Performance art                          5567   5.3%  ✅ mapped
       8  Dance                                    5012   4.8%  ✅ mapped
       9  Guitar                                   4208   4.0%  ✅ mapped
      10  String instrument                        3931   3.7%  ✅ mapped
      11  Food         

In [36]:
# ── Только новые кандидаты, не вошедшие в маппинг ────────────
print(f"🆕 Все незамапленные классы с 500+ видео:")
print(f"   {'ID':>5}  {'Название':<35}  {'Кол-во':>8}")
print("   " + "─" * 52)

for label_id, name, count in candidates_new:
    print(f"   {label_id:>5}  {name:<35}  {count:>8}")

🆕 Все незамапленные классы с 500+ видео:
      ID  Название                               Кол-во
   ────────────────────────────────────────────────────
      21  Mobile phone                             1964
      23  Smartphone                               1748
      24  Gadget                                   1705
      25  Trailer (promotion)                      1672
      27  Minecraft                                1543
      28  Drum kit                                 1535
      26  Toy                                      1525
      29  Cuisine                                  1510
      32  Dish (food)                              1490
      33  Drum                                     1478
      30  Motorcycle                               1476
      31  Piano                                    1432
      34  Acoustic guitar                          1394
      36  Call of Duty                             1330
      37  Electric guitar                          1273
      3